In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [1]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr
import scipy.stats as stats


In [2]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "sc_2023_Glyco/sc_myeloid"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "sc_2023_Glyco/sc_myeloid/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=800) # set sufficiently high resolution for saving 400dpi

In [ ]:
###--- Load pre-filtering data ---###

In [3]:
# Data Loading
adata_celltypist = sc.read("sc_myeloid_clustering.h5ad")
# Start with Raw data
adata_celltypist.X = adata_celltypist.layers["counts"].copy()
# Clean of variables
# List of samples
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata_celltypist.obs['sample'].isin(comparison_id)
adata_celltypist_my = adata_celltypist[boolean_mask, :]
del (adata_celltypist_my.uns['log1p'])

In [ ]:
###--- Generation of pseudo-bulk profiles ---###

In [ ]:
## Exploration of pseudobulk profiles ##

In [4]:
# Get filtered pseudo-bulk profile
pdata = dc.get_pseudobulk(adata_celltypist_my, sample_col='sample',
    groups_col='leiden', layer='counts',
    mode='sum', min_cells=10, min_counts=1000
)
pdata

AnnData object with n_obs × n_vars = 125 × 34273
    obs: 'sample', 'species', 'donor', 'label', 'predicted_doublets', 'leiden', 'psbulk_n_cells', 'psbulk_counts'
    var: 'gene_id', 'gene_name', 'genome', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    layers: 'psbulk_props'

In [ ]:
# Pseudo-bulk profile gene filtering

In [5]:
np.unique(adata_celltypist_my.obs['leiden'], return_counts=True)

(array(['0', '1', '10', '11', '12', '13', '14', '2', '3', '4', '5', '6',
        '7', '8', '9'], dtype=object),
 array([3014, 2735,  771,  820,  410,  167,   47, 2564, 1867, 1359, 1475,
        1395,  913,  614,  673]))

In [35]:
# Import DESeq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Subsetting the cell type of interest  ##
cluster_idex = '7'
sp_cells = pdata[pdata.obs['leiden'] == cluster_idex].copy()

# Obtain genes that pass the thresholds
#dc.plot_filter_by_expr(sp_cells, group='label', min_count=10, min_total_count=10)
genes = dc.filter_by_expr(sp_cells, group='label', min_count=5, min_total_count=15)

# Filter by these genes
sp_cells = sp_cells[:, genes].copy()
sp_cells

## Contrast between conditions ##

# Build DESeq2 object
dds = DeseqDataSet(
    adata=sp_cells,
    design_factors=['donor','label'],
    ref_level=['label', 'CAR'],
    refit_cooks=True,
    n_cpus=8,
)

# Compute LFCs
dds.deseq2()

# Extract contrast
comparison = 'Tr2DG'
stat_res = DeseqStats(dds, contrast=["label", comparison, 'CAR'], n_cpus=8, cooks_filter= False, independent_filter=False)
stat_res.summary()

# Shrink LFCs
stat_res.lfc_shrink()

# Extract results
results_df = stat_res.results_df
results_df
results_df.to_csv(f"pdeuso_myeloid/de_deseq2_myeloid_{cluster_idex}_cluster_vs{comparison}.csv")

dc.plot_volcano_df(results_df, x='log2FoldChange', y='padj', top=20, save= f"pdeuso_myeloid/de_deseq2_myeloid_{cluster_idex}_cluster_vs{comparison}.pdf")

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 8.91 seconds.

Fitting dispersion trend curve...
... done in 1.74 seconds.

Fitting MAP dispersions...
... done in 6.11 seconds.

Fitting LFCs...
... done in 0.81 seconds.

Refitting 0 outliers.

Running Wald tests...
... done in 0.39 seconds.

Log2 fold change & Wald test p-value: label Tr2DG vs CAR


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
7SK,6.125209,-1.019048,0.564106,-1.806482,0.070843,0.760155
AACS,3.906672,-0.563277,0.837141,-0.672858,0.501037,0.969698
AAGAB,6.707180,-0.320781,0.670221,-0.478620,0.632209,0.977783
AAK1,48.914833,-0.051945,0.201037,-0.258386,0.796109,0.995959
AATF,13.273415,0.238948,0.460426,0.518971,0.603781,0.977783
...,...,...,...,...,...,...
ZXDC,9.247069,0.001935,0.417383,0.004635,0.996302,0.998912
ZYG11B,4.509951,-0.822851,0.682461,-1.205712,0.227928,0.923404
ZYX,15.024577,0.750722,0.318862,2.354380,0.018554,0.588889
ZZEF1,24.763098,-0.078154,0.266548,-0.293208,0.769364,0.986400


Fitting MAP LFCs...
... done in 1.74 seconds.

Shrunk Log2 fold change & Wald test p-value: label Tr2DG vs CAR


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
7SK,6.125209,-0.144270,0.328404,-1.806482,0.070843,0.760155
AACS,3.906672,-0.012011,0.113752,-0.672858,0.501037,0.969698
AAGAB,6.707180,-0.010249,0.110618,-0.478620,0.632209,0.977783
AAK1,48.914833,-0.013932,0.099450,-0.258386,0.796109,0.995959
AATF,13.273415,0.015861,0.113553,0.518971,0.603781,0.977783
...,...,...,...,...,...,...
ZXDC,9.247069,-0.144270,0.328404,0.004635,0.996302,0.998912
ZYG11B,4.509951,-0.144270,0.328404,-1.205712,0.227928,0.923404
ZYX,15.024577,-0.144270,0.328404,2.354380,0.018554,0.588889
ZZEF1,24.763098,-0.144270,0.328404,-0.293208,0.769364,0.986400
